In [29]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import LabelEncoder
import soundfile as sf
import torchaudio

In [30]:
# Define the neural network
class GenreClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(GenreClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)  # Dropout layer
        
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.5)  # Dropout layer
        
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout1(x)
        x = self.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [31]:
# Load the trained genre classifier
model = GenreClassifier(input_size=518, num_classes=8)
model.load_state_dict(torch.load('genre_classifier.pth'))
model.eval()

# Load label encoder
label_encoder = LabelEncoder()
label_encoder.classes_ = np.load('genre_classes.npy', allow_pickle=True)  # Save/load class mapping

# Load input audio
y, sr = librosa.load("myaudio.mp3", sr=None)

In [32]:
# Convertir y (audio) en tenseur avec dimension batch
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)  # [1, time]
y_tensor = torch.nn.functional.normalize(y_tensor, dim=1)



In [ ]:
#calcul des MFCC

mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=sr,
    n_mfcc=20,
    melkwargs={"n_fft": 2048, "n_mels": 40}
)

def compute_mfccs_from_audio(audio_tensor):
    return mfcc_transform(audio_tensor)




In [34]:
# Initialisation de M comme paramètre optimisable
M = torch.nn.Parameter(torch.zeros_like(y_tensor), requires_grad=True)
optimizer = torch.optim.Adam([M], lr=0.01)


In [35]:
target_genre = "Electronic"
target_id = torch.tensor([label_encoder.transform([target_genre])[0]], dtype=torch.long)

num_steps = 100
for step in range(num_steps):
    optimizer.zero_grad()

    # Audio transformé (additif)
    y_transformed = y_tensor + M
    

    # MFCCs depuis l’audio modifié
    mfccs = compute_mfccs_from_audio(y_transformed)
    features = mfccs.flatten(start_dim=1)[:, :model.fc1.in_features]

    # Prédiction du modèle
    outputs = model(features)
    probs = torch.softmax(outputs, dim=1)

    # Perte : on veut augmenter la proba du genre cible
    loss = -torch.log(probs[0, target_id] + 1e-9)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        M.clamp_(-0.5, 0.5)  # Limite l'amplitude du masque

    print(f"Step {step+1}, Loss: {loss.item():.4f}")


Step 1, Loss: 20.7233
Step 2, Loss: 20.7233
Step 3, Loss: 20.7233
Step 4, Loss: 20.7233
Step 5, Loss: 20.7233
Step 6, Loss: 20.7233
Step 7, Loss: 20.7233
Step 8, Loss: 20.7233
Step 9, Loss: 20.7233
Step 10, Loss: 20.7233
Step 11, Loss: 20.7233
Step 12, Loss: 20.7233
Step 13, Loss: 20.7233
Step 14, Loss: 20.7233
Step 15, Loss: 20.7233
Step 16, Loss: 20.7233
Step 17, Loss: 20.7233
Step 18, Loss: 20.7233
Step 19, Loss: 20.7233
Step 20, Loss: 20.7233
Step 21, Loss: 20.7233
Step 22, Loss: 20.7233
Step 23, Loss: 20.7233
Step 24, Loss: 20.7233
Step 25, Loss: 20.7233
Step 26, Loss: 20.7233
Step 27, Loss: 20.7233
Step 28, Loss: 20.7233
Step 29, Loss: 20.7233
Step 30, Loss: 20.7233
Step 31, Loss: 20.7233
Step 32, Loss: 20.7233
Step 33, Loss: 20.7233
Step 34, Loss: 20.7233
Step 35, Loss: 20.7233
Step 36, Loss: 20.7233
Step 37, Loss: 20.7233
Step 38, Loss: 20.7233
Step 39, Loss: 20.7233
Step 40, Loss: 20.7233
Step 41, Loss: 20.7233
Step 42, Loss: 20.7233
Step 43, Loss: 20.7233
Step 44, Loss: 20.72

In [36]:
# Reconstruction : y_transformed final
final_audio = (y_tensor + M).detach().numpy().squeeze()
final_audio = librosa.util.normalize(final_audio)

# Sauvegarde dans un fichier audio
sf.write("output_transferred.wav", final_audio, sr)
print("Fichier audio transformé enregistré sous 'output_transferred.wav'")


Fichier audio transformé enregistré sous 'output_transferred.wav'
